<a href="https://colab.research.google.com/github/mehanshbarthwal-lab/search-ranking-ml/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mehanshbarthwal-lab/search-ranking-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Week 5 Data Loading & Preprocessing
This cell loads the dataset and prepares the variables (`df`, `X`, `y`, `groups`) required for the validation audit.

In [9]:
import duckdb
import pandas as pd

con = duckdb.connect()

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_Token')
except Exception:
    from getpass import getpass
    hf_token = getpass('HF_Token: ')

con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

FACT = "'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'"
DIM = "'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'"

query = f"""
WITH prior AS (
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS gsc_impressions_prior30,
           SUM(gsc_clicks) AS gsc_clicks_prior30,
           AVG(gsc_avg_position) AS gsc_avg_position_prior30
    FROM {FACT}
    WHERE month = '2026-02'
    GROUP BY content_hash_id, client_hash_id
),
last AS (
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS gsc_impressions_last30,
           SUM(gsc_clicks) AS gsc_clicks_last30
    FROM {FACT}
    WHERE month = '2026-03'
    GROUP BY content_hash_id, client_hash_id
)
SELECT
    p.content_hash_id, p.client_hash_id,
    p.gsc_impressions_prior30, p.gsc_clicks_prior30, p.gsc_avg_position_prior30,
    l.gsc_impressions_last30, l.gsc_clicks_last30,
    d.word_count, d.content_type, d.main_intent
FROM prior p
JOIN last l USING (content_hash_id, client_hash_id)
JOIN {DIM} d USING (content_hash_id)
WHERE p.gsc_impressions_prior30 >= 50 AND p.gsc_clicks_prior30 >= 3
"""

df = con.sql(query).df()
print(f"Rows: {len(df):,}")

df['impr_change_pct'] = (df['gsc_impressions_last30'] - df['gsc_impressions_prior30']) / df['gsc_impressions_prior30'] * 100
df['click_change_pct'] = (df['gsc_clicks_last30'] - df['gsc_clicks_prior30']) / df['gsc_clicks_prior30'] * 100

def assign_pattern(row):
    if row['impr_change_pct'] >= -5 and row['click_change_pct'] <= -15:
        return 'answered_away'
    elif row['impr_change_pct'] <= -15 and row['click_change_pct'] <= -15:
        return 'normal_decay'
    else:
        return 'stable_other'

df['pattern_group'] = df.apply(assign_pattern, axis=1)
df['gsc_ctr_prior30'] = df['gsc_clicks_prior30'] / df['gsc_impressions_prior30']

print(df['pattern_group'].value_counts())
print(f"\nBase rate (answered_away): {(df['pattern_group'] == 'answered_away').mean():.3f}")

y = (df['pattern_group'] == 'answered_away').astype(int)

feature_cols_numeric = [
    'gsc_impressions_prior30', 'gsc_clicks_prior30', 'gsc_avg_position_prior30',
    'word_count', 'gsc_ctr_prior30'
]
feature_cols_categorical = ['content_type', 'main_intent']

X = pd.get_dummies(df[feature_cols_numeric + feature_cols_categorical], dummy_na=True)
X[feature_cols_numeric] = X[feature_cols_numeric].fillna(0)

groups = df['client_hash_id']

print(f"\nX shape: {X.shape}, y length: {len(y)}, groups length: {len(groups)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 29,516
pattern_group
stable_other     19186
answered_away     6759
normal_decay      3571
Name: count, dtype: int64

Base rate (answered_away): 0.229

X shape: (29516, 14), y length: 29516, groups length: 29516


## 1. Two paper findings + my methodology questions

**Finding #1 — "The Freshness Multiplier" (361+ day bucket, 283:1 growth ratio)**

Where does the number come from: the paper computes a growth-to-decline ratio per freshness
bucket, and the 361+ bucket shows 283 growing pages against just 1 declining page. My question:
with n=1 on the declining side, is 283:1 a stable measurement or is it one page away from being
a completely different number? The paper actually answers this itself in the same section,
noting the bucket is "tiny and unstable," which is exactly the kind of self-correction this
assignment asks me to practice — so my methodology question here is really "would I have caught
this without the paper flagging it for me," and honestly I'm not sure I would have on a first
pass. The stricter number to report is probably a confidence interval on that ratio, or just
dropping the headline figure and keeping the 31-90 day bucket's 7.88:1 as the real claim, which
is what the paper does.

**Finding #2 — "AI Model Performance" (OpenAI vs Gemini age-controlled cohorts)**

Where does the label come from: health score, which the paper is upfront is a FlyRank composite
(impressions + position + CTR + scroll depth), not an external outcome. My methodology question:
does the validation design support a claim about which AI provider produces better content, or
does it only support a claim about which provider's pages score higher on a metric that's
partly made of the same inputs (impressions, position) that also drive provider assignment
indirectly through publish timing and topic mix? The paper's own random forest appendix shows
Average Position and Impressions are 43% and 32% of what predicts health score, so two providers
differing on health score could partly just mean they differ on position and impressions, which
is closer to circular than causal. The paper does hedge this correctly ("not a victory lap for
one provider family"), so my question is more about whether a reader skimming just the chart
would pick up that hedge, or just see "Gemini leads" and stop there.

## 2. My model under an honest split (before/after)

Before/after: random split vs grouped split, same features, same models, from Week 5

I already ran this comparison in Week 5, so this section re-runs and audits it rather than
re-inventing it — the "before" is the random train_test_split(..., stratify=y, random_state=42)
Week 3 also used, and the "after" is GroupShuffleSplit grouped by client_hash_id, so no client
appears in both train and test.

Results (F1 on the answered_away minority class, base rate 0.229):

| model | random split | grouped split | gap |
|---|---|---|---|
| Logistic Regression | 0.365 | 0.350 | 0.015 |
| Decision Tree | 0.057 | 0.014 | 0.043 |
| Random Forest | 0.136 | 0.110 | 0.026 |

Both splits clear Week 3's honest baseline F1 of 0.135 for Logistic Regression on both splits, and
Random Forest clears it narrowly on the random split (0.136) but falls just under it on the
grouped split (0.110). The random-to-grouped gap is small across all three models (0.015-0.043),
a different story than the framework video's own forest example (0.996 random down to 0.496
grouped) — that example had a much bigger drop, suggesting that model was leaning on something
client-specific. My gap is a weaker but real signal this feature set isn't leaning heavily on
memorized client identity, though 22 training clients and 8 test clients is still a small number
of groups to generalize from, so I'm stating this as observed on this split, not proven stable
across all possible client splits.

Note: re-running this cell produced slightly different random-split numbers than my first Week 5
run (0.365 vs 0.370 for Logistic Regression, 0.136 vs 0.149 for Random Forest), while the grouped
split numbers matched almost exactly. The likely cause is that DuckDB doesn't guarantee row order
without an explicit ORDER BY, so df can assemble in a different row order between sessions, which
shifts what train_test_split sees before its random_state=42 shuffle takes over. The grouped
split staying stable while the random split doesn't is itself informative — it's one more small
piece of evidence that the grouped result is the more trustworthy number to report, since it's
less sensitive to an incidental detail like row order.

Decision Tree collapsing to near-zero F1 on both splits is expected given a 22.9% minority class
and no class weighting on the tree — I'm not troubleshooting that here since Logistic Regression
and Random Forest are the models actually being compared to baseline.

In [10]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

WEEK3_HONEST_F1 = 0.135
BASE_RATE = round(y.mean(), 3)

def run_and_score(model, X_tr, X_te, y_tr, y_te, name, split_name, scale=False):
    if scale:
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_te = scaler.transform(X_te)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    f1 = f1_score(y_te, preds)
    return {'model': name, 'split': split_name, 'f1_answered_away': round(f1, 3)}

results = []

X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
results.append(run_and_score(LogisticRegression(max_iter=1000, class_weight='balanced'),
                              X_train_rand, X_test_rand, y_train_rand, y_test_rand,
                              'Logistic Regression', 'random', scale=True))
results.append(run_and_score(DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=42),
                              X_train_rand, X_test_rand, y_train_rand, y_test_rand, 'Decision Tree', 'random'))
results.append(run_and_score(RandomForestClassifier(n_estimators=200, random_state=42),
                              X_train_rand, X_test_rand, y_train_rand, y_test_rand, 'Random Forest', 'random'))

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
overlap = train_clients & test_clients
print(f"Client overlap in grouped split: {len(overlap)} (should be 0)")
assert len(overlap) == 0, "Grouped split leaked a client across train/test"

results.append(run_and_score(LogisticRegression(max_iter=1000, class_weight='balanced'),
                              X_train_grp, X_test_grp, y_train_grp, y_test_grp,
                              'Logistic Regression', 'grouped', scale=True))
results.append(run_and_score(DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=42),
                              X_train_grp, X_test_grp, y_train_grp, y_test_grp, 'Decision Tree', 'grouped'))
results.append(run_and_score(RandomForestClassifier(n_estimators=200, random_state=42),
                              X_train_grp, X_test_grp, y_train_grp, y_test_grp, 'Random Forest', 'grouped'))

results_df = pd.DataFrame(results)
print(f"\nBase rate (answered_away): {BASE_RATE}")
print(f"Week 3 honest F1 baseline (reference, random split): {WEEK3_HONEST_F1}")
print()
print(results_df.to_string(index=False))

pivot = results_df.pivot(index='model', columns='split', values='f1_answered_away')
pivot['gap'] = (pivot['random'] - pivot['grouped']).round(3)
print("\nRandom-to-grouped gap by model:")
print(pivot)

Client overlap in grouped split: 0 (should be 0)

Base rate (answered_away): 0.229
Week 3 honest F1 baseline (reference, random split): 0.135

              model   split  f1_answered_away
Logistic Regression  random             0.373
      Decision Tree  random             0.000
      Random Forest  random             0.158
Logistic Regression grouped             0.350
      Decision Tree grouped             0.014
      Random Forest grouped             0.107

Random-to-grouped gap by model:
split                grouped  random    gap
model                                      
Decision Tree          0.014   0.000 -0.014
Logistic Regression    0.350   0.373  0.023
Random Forest          0.107   0.158  0.051


## 3. Leakage audit

Running the attack checklist from `hunting-leakage-and-validating` against my final Week 5
feature set: `gsc_impressions_prior30`, `gsc_clicks_prior30`, `gsc_avg_position_prior30`,
`word_count`, `gsc_ctr_prior30`, plus one-hot `content_type` and `main_intent`.

- **Timeline**: all five numeric features are prior-30-day window aggregates or static content
  properties, computed strictly before the last-30-day label window that defines
  `pattern_group`. None overlap the label window.
- **Label-derived / sibling columns**: `click_change_pct` (the actual leaky feature from Week 3)
  is confirmed absent from the final feature list. `gsc_ctr_prior30` is a ratio of two prior-
  window columns only (`gsc_clicks_prior30 / gsc_impressions_prior30`), so it doesn't touch
  `gsc_clicks_last30` or `gsc_impressions_last30` at all, which is what makes it pass where
  `click_change_pct` failed.
- **Product flags / existing-system scores**: none of the FlyRank Health Score, Optimization
  Flags, or Trend Direction columns are in the feature set. `pattern_group` itself is only ever
  used to build the label `y`, never as a feature — confirmed by checking `X.columns` doesn't
  contain it.
- **Grouped split**: done in Section 2, `client_hash_id` grouping, zero client overlap asserted.
- **Base rate printed next to every metric**: yes, 0.229 for answered_away, referenced in
  Section 2 next to every F1 score.
- **Top feature importance sanity check**: `gsc_avg_position_prior30` (0.234),
  `gsc_impressions_prior30` (0.229), and `gsc_ctr_prior30` (0.227) come out roughly tied as the
  top three in the Random Forest, with none towering over the others the way a leaked feature
  would (Week 3's `click_change_pct` before removal had that towering pattern — this doesn't).
  I'm treating this as a passed sanity check, not a celebration.
- **Note on `content_age_days`**: Section 1 of my Week 5 notebook describes adding
  `content_age_days` as a new feature, but the actual `feature_cols_numeric` list used in the
  model only contains the five features listed above — `content_age_days` isn't there. This is a
  documentation/code mismatch, not a leakage problem (the feature I described would have passed
  the decision-time test if it had been included), but I'm noting it here because catching gaps
  between what I said I did and what the code actually does is the exact muscle this assignment
  is asking me to build.

I also ran the deliberate-leak check the skill recommends: adding `click_change_pct` back into
the feature set and re-training, to confirm my test harness actually catches leakage rather than
just assuming it does.

In [11]:
# Deliberate-leak check: add back the known-leaky feature and confirm the score jumps.
# If it doesn't jump, the test harness itself can't be trusted.

df['click_change_pct'] = (
    (df['gsc_clicks_last30'] - df['gsc_clicks_prior30']) / df['gsc_clicks_prior30'] * 100
)

leaky_numeric = ['gsc_impressions_prior30', 'gsc_clicks_prior30', 'gsc_avg_position_prior30',
                  'word_count', 'gsc_ctr_prior30', 'click_change_pct']
X_leaky = pd.get_dummies(df[leaky_numeric + feature_cols_categorical], dummy_na=True)
X_leaky[leaky_numeric] = X_leaky[leaky_numeric].fillna(0)

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_leaky, y, test_size=0.25, random_state=42, stratify=y
)

rf_leaky = RandomForestClassifier(n_estimators=200, random_state=42)
rf_leaky.fit(X_train_leak, y_train_leak)
preds_leaky = rf_leaky.predict(X_test_leak)
f1_leaky = f1_score(y_test_leak, preds_leaky)

# Compare against the honest random-forest random-split score from Section 2
honest_rf_random = results_df.query("model == 'Random Forest' and split == 'random'")['f1_answered_away'].iloc[0]

print(f"Honest Random Forest F1 (random split, no leak): {honest_rf_random}")
print(f"Random Forest F1 WITH click_change_pct added back in: {round(f1_leaky, 3)}")
print(f"Jump: {round(f1_leaky - honest_rf_random, 3)}")
print()
if f1_leaky > honest_rf_random + 0.1:
    print("Test harness confirmed working: adding a known-leaky feature produces a large jump,")
    print("same pattern as Week 3. Dropping click_change_pct again and keeping the honest features.")
else:
    print("No large jump — this would mean the test harness isn't sensitive to leakage and needs")
    print("investigation before trusting any other number in this notebook.")

# Feature importance check on the leaky version — confirms it's the leak driving the jump
importances_leaky = pd.Series(rf_leaky.feature_importances_, index=X_leaky.columns).sort_values(ascending=False)
print("\nTop 5 feature importances with the leak present:")
print(importances_leaky.head(5))

Honest Random Forest F1 (random split, no leak): 0.158
Random Forest F1 WITH click_change_pct added back in: 0.703
Jump: 0.545

Test harness confirmed working: adding a known-leaky feature produces a large jump,
same pattern as Week 3. Dropping click_change_pct again and keeping the honest features.

Top 5 feature importances with the leak present:
click_change_pct            0.524224
gsc_avg_position_prior30    0.110038
gsc_impressions_prior30     0.109319
gsc_ctr_prior30             0.099985
word_count                  0.082298
dtype: float64


## 4. Claim rewrite

My boldest sentence, from Week 5's writeup:

Original: "Both [Logistic Regression and Random Forest] clear the honest F1 of 0.135 by a wide
margin on both splits — 0.370 random, 0.350 grouped."

Using the claim ladder from writing-honest-claims, this sentence sits between "a measured
comparison" and "a validated model that ranks/predicts out-of-sample" — I have a grouped,
out-of-sample split behind it, so it's not overclaiming into causal language, but "wide margin"
is doing some unearned drama work that the actual numbers don't need.

Rewrite: "On a client-grouped holdout, Logistic Regression scored F1=0.350 on the answered_away
class, against a base rate of 0.229 and Week 3's honest baseline of 0.135 — an observed
improvement over baseline on data the model's training clients never saw. Random Forest scored
F1=0.110 on the same grouped holdout, below Week 3's own Random Forest reference of 0.135, so the
improvement over baseline is specific to Logistic Regression in this comparison, not a property
of every model tried."

What changed: dropped "wide margin" (a drama word, not a measured one), named the actual split
type instead of leaving it implicit, put the base rate directly next to the F1 the way the
skill's checklist requires, and stopped implying all models improved when only two of three did
and one of those two only barely cleared the bar depending on which split you read. This is
decision-support language, not causal — it says the model looks worth using to help prioritize a
review queue, not that it will improve outcomes, since no A/B test sits behind any of this.

In [12]:
# No new computation needed — this section rewrites language, not numbers.
# Re-printing the exact figures the rewrite above depends on, so the claim and its
# evidence sit next to each other in the executed notebook.

print("Figures backing the Section 4 claim rewrite:")
print(f"Base rate (answered_away): {BASE_RATE}")
print(f"Week 3 honest F1 baseline: {WEEK3_HONEST_F1}")
print(results_df.query("model in ['Logistic Regression', 'Random Forest']").to_string(index=False))

Figures backing the Section 4 claim rewrite:
Base rate (answered_away): 0.229
Week 3 honest F1 baseline: 0.135
              model   split  f1_answered_away
Logistic Regression  random             0.373
      Random Forest  random             0.158
Logistic Regression grouped             0.350
      Random Forest grouped             0.107


## 5. Threshold tuning — can a better cutoff beat F1=0.350 without new data?

Everything above used .predict(), which defaults to a 0.5 probability cutoff for classifying a
page as answered_away. With a 22.9% minority class, 0.5 was never a principled choice — it's just
sklearn's default. This section sweeps thresholds on the grouped-split Logistic Regression model
(the current best performer) using predict_proba() instead, and reports F1 at each threshold to
find whether a different cutoff does better than the default, using only the model already
trained above — no new features, no new data, no re-fitting.

I'm doing this only on the grouped split, since that's the split I've been treating as the honest
one throughout this notebook, and only on Logistic Regression, since Random Forest already fell
below Week 3's own baseline on this split and tuning its threshold wouldn't change the more
important finding that it's the weaker model here.

In [13]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

# Reuse the already-trained grouped-split Logistic Regression setup from Section 2.
# Re-fit here explicitly so this section works standalone even if run out of order.
scaler_thresh = StandardScaler()
X_train_grp_scaled = scaler_thresh.fit_transform(X_train_grp)
X_test_grp_scaled = scaler_thresh.transform(X_test_grp)

logreg_thresh = LogisticRegression(max_iter=1000, class_weight='balanced')
logreg_thresh.fit(X_train_grp_scaled, y_train_grp)

# Probability of the positive class (answered_away = 1), not the hard 0/1 label
probs = logreg_thresh.predict_proba(X_test_grp_scaled)[:, 1]

thresholds = np.arange(0.10, 0.95, 0.05)
threshold_results = []

for t in thresholds:
    preds_t = (probs >= t).astype(int)
    f1_t = f1_score(y_test_grp, preds_t, zero_division=0)
    precision_t = precision_score(y_test_grp, preds_t, zero_division=0)
    recall_t = recall_score(y_test_grp, preds_t, zero_division=0)
    threshold_results.append({
        'threshold': round(t, 2),
        'f1': round(f1_t, 3),
        'precision': round(precision_t, 3),
        'recall': round(recall_t, 3),
        'flagged_count': int(preds_t.sum())
    })

threshold_df = pd.DataFrame(threshold_results)
print("Threshold sweep — Logistic Regression, grouped split:")
print(threshold_df.to_string(index=False))

best_row = threshold_df.loc[threshold_df['f1'].idxmax()]
default_row = threshold_df.loc[threshold_df['threshold'] == 0.50]

print(f"\nDefault threshold (0.50): F1={default_row['f1'].values[0]}")
print(f"Best threshold found ({best_row['threshold']}): F1={best_row['f1']}")
print(f"Improvement over default: {round(best_row['f1'] - default_row['f1'].values[0], 3)}")

# Precision@50 — a more decision-relevant metric if a content editor can only
# realistically review 50 pages a week, regardless of what threshold that implies
sorted_idx = np.argsort(-probs)
top_50_idx = sorted_idx[:50]
precision_at_50 = y_test_grp.values[top_50_idx].mean()
print(f"\nPrecision@50 (top 50 pages by predicted probability, any threshold): {round(precision_at_50, 3)}")
print("This answers a different, arguably more useful question: of the top 50 pages this model")
print("would tell an editor to review first, how many are actually answered_away?")

Threshold sweep — Logistic Regression, grouped split:
 threshold    f1  precision  recall  flagged_count
      0.10 0.394      0.245   0.999          15323
      0.15 0.394      0.245   0.999          15306
      0.20 0.394      0.246   0.998          15277
      0.25 0.394      0.246   0.993          15175
      0.30 0.393      0.246   0.973          14853
      0.35 0.389      0.246   0.929          14191
      0.40 0.386      0.248   0.870          13166
      0.45 0.381      0.259   0.725          10532
      0.50 0.350      0.282   0.462           6170
      0.55 0.295      0.296   0.293           3723
      0.60 0.252      0.303   0.215           2672
      0.65 0.134      0.351   0.083            883
      0.70 0.019      0.569   0.010             65
      0.75 0.008      0.625   0.004             24
      0.80 0.001      0.500   0.001              4
      0.85 0.000      0.000   0.000              0
      0.90 0.000      0.000   0.000              0

Default threshold (0.50): F

## 6. Adding candidate features, with leakage check and feature selection

Four new candidate features, all static content properties or prior-known metadata — none
derived from the last-30-day label window, so all pass the same decision-time test the existing
five features already passed:

- content_age_days — planned in Week 5, never actually added to the feature set. Free addition.
- days_since_update — content freshness, static and known ahead of the decision date.
- competition — keyword-difficulty metadata, not derived from any outcome window.
- search_volume — keyword-volume metadata. The FlyRank paper itself found this barely correlates
  with real performance (r=0.008 raw, -0.042 log-scaled), so I expect this one to add little or
  get dropped by feature selection, but I'm testing it rather than assuming that ahead of time.

Before trusting any of these, I run the same attack checklist from Section 3: confirm none of
them touch the last-30-day columns, then run a leakage sanity check on feature importance the
same way, looking for any single new feature that towers over the rest the way click_change_pct
did.

After the leakage check, I run feature selection (permutation importance on the grouped-split
Logistic Regression, which the training-honest-models skill recommends for "what drives X"
questions) to see which of the four candidates actually earn a place, rather than keeping all
four just because they passed the leakage test. Passing the leakage test means a feature is safe
to use — it doesn't mean it's useful.

In [16]:
import duckdb
import pandas as pd

con2 = duckdb.connect()
con2.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

schema_check = con2.sql(f"DESCRIBE SELECT * FROM {DIM}").df()
print("Actual dim_content columns:")
print(schema_check.to_string(index=False))

# Decision date matches Week 3-5's convention: prior30 = Feb 2026, last30 = March 2026
query_new_features = f"""
SELECT
    content_hash_id,
    DATE_DIFF('day', content_created_date, DATE '2026-03-31') AS content_age_days,
    DATE_DIFF('day', content_updated_date, DATE '2026-03-31') AS days_since_update
FROM {DIM}
"""
new_features_df = con2.sql(query_new_features).df()

df_extended = df.merge(new_features_df, on='content_hash_id', how='left')
print(f"Rows after merge: {len(df_extended):,} (should still be {len(df):,})")
print(f"\nNull counts in new columns:")
print(df_extended[['content_age_days', 'days_since_update']].isna().sum())

new_feature_cols = ['content_age_days', 'days_since_update']

# Leakage sanity check: confirm neither column name suggests a last-30-window origin
overlap_check = [c for c in new_feature_cols if 'last30' in c.lower() or 'last_30' in c.lower()]
print(f"\nColumns with suspicious last30-window naming: {overlap_check} (should be empty)")


from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

feature_cols_numeric_extended = feature_cols_numeric + new_feature_cols
X_extended = pd.get_dummies(
    df_extended[feature_cols_numeric_extended + feature_cols_categorical], dummy_na=True
)
X_extended[feature_cols_numeric_extended] = X_extended[feature_cols_numeric_extended].fillna(0)

groups_extended = df_extended['client_hash_id']
gss_ext = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx_ext, test_idx_ext = next(gss_ext.split(X_extended, y, groups=groups_extended))
X_train_ext, X_test_ext = X_extended.iloc[train_idx_ext], X_extended.iloc[test_idx_ext]
y_train_ext, y_test_ext = y.iloc[train_idx_ext], y.iloc[test_idx_ext]

rf_ext = RandomForestClassifier(n_estimators=200, random_state=42)
rf_ext.fit(X_train_ext, y_train_ext)

importance_ext_df = pd.DataFrame({
    'feature': X_extended.columns,
    'rf_importance': rf_ext.feature_importances_
}).sort_values('rf_importance', ascending=False)

print("Feature importances with content_age_days and days_since_update included:")
print(importance_ext_df.head(15).to_string(index=False))
print()
print("Leakage sanity check: neither new feature should tower over the existing five the way")
print("click_change_pct did in Section 3 (that one hit ~0.52 importance alone). If either")
print("content_age_days or days_since_update shows something similarly dominant, investigate")
print("before trusting it.")

from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

scaler_ext = StandardScaler()
X_train_ext_scaled = scaler_ext.fit_transform(X_train_ext)
X_test_ext_scaled = scaler_ext.transform(X_test_ext)

logreg_ext = LogisticRegression(max_iter=1000, class_weight='balanced')
logreg_ext.fit(X_train_ext_scaled, y_train_ext)
preds_ext = logreg_ext.predict(X_test_ext_scaled)
f1_ext = f1_score(y_test_ext, preds_ext)

print(f"Logistic Regression F1, grouped split, ORIGINAL 5 features: 0.350 (from Section 2)")
print(f"Logistic Regression F1, grouped split, EXTENDED {len(feature_cols_numeric_extended)} features: {round(f1_ext, 3)}")
print(f"Change: {round(f1_ext - 0.350, 3)}")

perm_result = permutation_importance(
    logreg_ext, X_test_ext_scaled, y_test_ext, n_repeats=20, random_state=42, scoring='f1'
)
perm_df = pd.DataFrame({
    'feature': X_extended.columns,
    'perm_importance_mean': perm_result.importances_mean,
    'perm_importance_std': perm_result.importances_std
}).sort_values('perm_importance_mean', ascending=False)

print("\nPermutation importance (drop in F1 when feature is shuffled — higher means more useful):")
print(perm_df.to_string(index=False))

print("\nFeatures whose permutation importance is near zero or negative contributed little to")
print("nothing on this split — safe to drop even though they passed the leakage check, since")
print("passing leakage and being useful are different questions.")

Actual dim_content columns:
               column_name column_type null  key default extra
            client_hash_id     VARCHAR  YES None    None  None
           content_hash_id     VARCHAR  YES None    None  None
           keyword_hash_id     VARCHAR  YES None    None  None
               url_hash_id     VARCHAR  YES None    None  None
        keyword_char_count      BIGINT  YES None    None  None
       keyword_token_count      BIGINT  YES None    None  None
            url_char_count      BIGINT  YES None    None  None
      content_created_date        DATE  YES None    None  None
      content_updated_date        DATE  YES None    None  None
              content_type     VARCHAR  YES None    None  None
             search_volume      BIGINT  YES None    None  None
               competition      DOUBLE  YES None    None  None
         competition_level     VARCHAR  YES None    None  None
                       cpc      DOUBLE  YES None    None  None
               main_intent 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows after merge: 29,516 (should still be 29,516)

Null counts in new columns:
content_age_days     0
days_since_update    0
dtype: int64

Columns with suspicious last30-window naming: [] (should be empty)
Feature importances with content_age_days and days_since_update included:
                        feature  rf_importance
        gsc_impressions_prior30       0.181247
       gsc_avg_position_prior30       0.179755
                gsc_ctr_prior30       0.174426
                     word_count       0.139276
               content_age_days       0.124831
             gsc_clicks_prior30       0.093368
              days_since_update       0.070098
      main_intent_informational       0.011409
         main_intent_commercial       0.010609
      main_intent_transactional       0.010387
                main_intent_nan       0.001569
   content_type_keyword article       0.001244
    content_type_feedly article       0.001209
       main_intent_navigational       0.000495
content_type_co

## 7. Testing gradient boosting — was skipping it in Week 5 the right call?

Week 5 explicitly held off on gradient boosting, reasoning that with only 5-7 features and
~29.5k rows, the added complexity hadn't been earned yet, and that boosting should wait until
simpler models show there's real signal worth chasing. Logistic Regression at F1=0.350
(grouped split) is that signal. This section tests whether XGBoost actually improves on that
number, using the same grouped split, same original 5-feature set from Section 2 (not the
extended 7-feature set from Section 6, since that set was already shown not to help), and the
same F1-on-answered_away metric throughout.

This is a check on an earlier decision, not an assumed upgrade — if XGBoost doesn't beat 0.350,
that's a real finding supporting the Week 5 call to hold off, not a failed experiment.

In [18]:
!pip install xgboost --quiet

import xgboost as xgb
from sklearn.metrics import f1_score

# Same grouped split, same original 5-feature X from Section 2 — no new features here,
# isolating the question to "does the model family matter" rather than mixing in Section 6's
# already-rejected feature additions
xgb_clf = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=(y_train_grp == 0).sum() / (y_train_grp == 1).sum(),  # handles class imbalance, XGBoost's equivalent of class_weight='balanced'
    random_state=42,
    eval_metric='logloss'
)

xgb_clf.fit(X_train_grp, y_train_grp)
xgb_preds = xgb_clf.predict(X_test_grp)
xgb_f1 = f1_score(y_test_grp, xgb_preds)

print(f"Logistic Regression F1, grouped split (Section 2, reference): 0.350")
print(f"Random Forest F1, grouped split (Section 2, reference): 0.110")
print(f"XGBoost F1, grouped split: {round(xgb_f1, 3)}")
print(f"XGBoost vs Logistic Regression: {round(xgb_f1 - 0.350, 3)}")

xgb_precision = precision_score(y_test_grp, xgb_preds, zero_division=0)
xgb_recall = recall_score(y_test_grp, xgb_preds, zero_division=0)
print(f"\nXGBoost precision: {round(xgb_precision, 3)}, recall: {round(xgb_recall, 3)}")

import numpy as np
import pandas as pd

xgb_importance_df = pd.DataFrame({
    'feature': X.columns,
    'xgb_importance': xgb_clf.feature_importances_
}).sort_values('xgb_importance', ascending=False)

print("XGBoost feature importances, grouped split:")
print(xgb_importance_df.to_string(index=False))

# Precision@50, same as Section 5's Logistic Regression check, for a direct comparison
xgb_probs = xgb_clf.predict_proba(X_test_grp)[:, 1]
sorted_idx_xgb = np.argsort(-xgb_probs)
top_50_idx_xgb = sorted_idx_xgb[:50]
xgb_precision_at_50 = y_test_grp.values[top_50_idx_xgb].mean()

print(f"\nXGBoost precision@50: {round(xgb_precision_at_50, 3)}")
print(f"Logistic Regression precision@50 (Section 5, reference): 0.580")

Logistic Regression F1, grouped split (Section 2, reference): 0.350
Random Forest F1, grouped split (Section 2, reference): 0.110
XGBoost F1, grouped split: 0.352
XGBoost vs Logistic Regression: 0.002

XGBoost precision: 0.283, recall: 0.465
XGBoost feature importances, grouped split:
                        feature  xgb_importance
        gsc_impressions_prior30        0.135094
       gsc_avg_position_prior30        0.124587
                     word_count        0.108343
                gsc_ctr_prior30        0.093732
                main_intent_nan        0.085896
   content_type_keyword article        0.077698
      main_intent_transactional        0.074332
             gsc_clicks_prior30        0.066693
    content_type_feedly article        0.066659
      main_intent_informational        0.058872
       main_intent_navigational        0.054950
         main_intent_commercial        0.053144
content_type_comparison article        0.000000
               content_type_nan        0.0

## 8. Hyperparameter tuning — can the models we already have do better, tuned properly?

Every model so far ran with hand-picked defaults, not a systematic search. This section tunes
Logistic Regression and XGBoost — the two real contenders, since Random Forest lost clearly in
Section 2 and Decision Tree collapsed — using GridSearchCV with GroupKFold instead of a plain
KFold, so the tuning process itself respects the same client-grouping honesty as every other
split in this notebook. Tuning on a random split would let the search quietly reward
client-memorization, which is exactly the problem Section 2 was built to catch — tuning has to
be held to the same standard as evaluation, or the resulting "best" hyperparameters would be
best at leaking, not best at the actual task.

I'm using 4-fold GroupKFold on the training portion of the grouped split only (X_train_grp,
y_train_grp), then checking the tuned model once on the held-out X_test_grp — never letting the
test set touch the search itself, so the final number stays a true out-of-sample estimate rather
than a number the search was indirectly optimizing toward.

In [19]:
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, make_scorer

f1_scorer = make_scorer(f1_score, pos_label=1)

# Scale once, upfront, since Logistic Regression needs it and GridSearchCV will refit repeatedly
scaler_tune = StandardScaler()
X_train_grp_scaled_tune = scaler_tune.fit_transform(X_train_grp)
X_test_grp_scaled_tune = scaler_tune.transform(X_test_grp)

param_grid_logreg = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']  # supports both l1 and l2, unlike the default solver
}

gkf = GroupKFold(n_splits=4)

grid_logreg = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    param_grid_logreg,
    scoring=f1_scorer,
    cv=gkf,
    n_jobs=-1
)

# GroupKFold needs the groups for the TRAINING portion only
groups_train_grp = groups.iloc[train_idx]

grid_logreg.fit(X_train_grp_scaled_tune, y_train_grp, groups=groups_train_grp)

print(f"Best Logistic Regression params: {grid_logreg.best_params_}")
print(f"Best cross-val F1 (on training folds only): {round(grid_logreg.best_score_, 3)}")

# Final check — once, on the untouched test set
tuned_logreg_preds = grid_logreg.best_estimator_.predict(X_test_grp_scaled_tune)
tuned_logreg_f1 = f1_score(y_test_grp, tuned_logreg_preds)

print(f"\nTuned Logistic Regression F1 on held-out test set: {round(tuned_logreg_f1, 3)}")
print(f"Untuned Logistic Regression F1 (Section 2, reference): 0.350")
print(f"Change: {round(tuned_logreg_f1 - 0.350, 3)}")

import xgboost as xgb

param_grid_xgb = {
    'max_depth': [3, 4, 6],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300],
    'min_child_weight': [1, 5]
}

scale_pos_weight_val = (y_train_grp == 0).sum() / (y_train_grp == 1).sum()

# Random search instead of full grid — 3x3x3x2 = 54 combos x 4 folds = 216 fits on full
# grid search, which is a lot for a dataset this size; randomized search samples a subset
from sklearn.model_selection import RandomizedSearchCV

random_search_xgb = RandomizedSearchCV(
    xgb.XGBClassifier(
        scale_pos_weight=scale_pos_weight_val,
        random_state=42,
        eval_metric='logloss'
    ),
    param_grid_xgb,
    n_iter=20,
    scoring=f1_scorer,
    cv=gkf,
    random_state=42,
    n_jobs=-1
)

random_search_xgb.fit(X_train_grp, y_train_grp, groups=groups_train_grp)

print(f"Best XGBoost params: {random_search_xgb.best_params_}")
print(f"Best cross-val F1 (on training folds only): {round(random_search_xgb.best_score_, 3)}")

tuned_xgb_preds = random_search_xgb.best_estimator_.predict(X_test_grp)
tuned_xgb_f1 = f1_score(y_test_grp, tuned_xgb_preds)

print(f"\nTuned XGBoost F1 on held-out test set: {round(tuned_xgb_f1, 3)}")
print(f"Untuned XGBoost F1 (Section 7, reference): 0.352")
print(f"Change: {round(tuned_xgb_f1 - 0.352, 3)}")

import pandas as pd

final_summary = pd.DataFrame([
    {'model': 'Decision Tree', 'variant': 'untuned', 'f1_grouped': 0.014},
    {'model': 'Random Forest', 'variant': 'untuned', 'f1_grouped': 0.110},
    {'model': 'Logistic Regression', 'variant': 'untuned', 'f1_grouped': 0.350},
    {'model': 'XGBoost', 'variant': 'untuned', 'f1_grouped': 0.352},
    {'model': 'Logistic Regression', 'variant': 'tuned (GroupKFold search)', 'f1_grouped': round(tuned_logreg_f1, 3)},
    {'model': 'XGBoost', 'variant': 'tuned (GroupKFold search)', 'f1_grouped': round(tuned_xgb_f1, 3)},
]).sort_values('f1_grouped', ascending=False)

print("Full model comparison, grouped split, F1 on answered_away:")
print(final_summary.to_string(index=False))

best_overall = final_summary.iloc[0]
print(f"\nBest model overall: {best_overall['model']} ({best_overall['variant']}), F1={best_overall['f1_grouped']}")

Best Logistic Regression params: {'C': 0.01, 'penalty': 'l2', 'solver': 'liblinear'}
Best cross-val F1 (on training folds only): 0.285

Tuned Logistic Regression F1 on held-out test set: 0.349
Untuned Logistic Regression F1 (Section 2, reference): 0.350
Change: -0.001
Best XGBoost params: {'n_estimators': 100, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.2}
Best cross-val F1 (on training folds only): 0.282

Tuned XGBoost F1 on held-out test set: 0.348
Untuned XGBoost F1 (Section 7, reference): 0.352
Change: -0.004
Full model comparison, grouped split, F1 on answered_away:
              model                   variant  f1_grouped
            XGBoost                   untuned       0.352
Logistic Regression                   untuned       0.350
Logistic Regression tuned (GroupKFold search)       0.349
            XGBoost tuned (GroupKFold search)       0.348
      Random Forest                   untuned       0.110
      Decision Tree                   untuned       0.014

B

## 9. Checking client diversity — is 30 clients all there is?

Every grouped-split result in this notebook has relied on the ~30 clients pulled by Week 3's
query, split roughly 22 training / 8 test. That's a small number of groups to base a
generalization claim on. Before accepting that as a hard limit, this section checks whether the
warehouse actually contains more clients than the current query's filters allow through —
specifically, whether the impressions_prior30 >= 50 and clicks_prior30 >= 3 thresholds are
excluding entire clients rather than just individual low-traffic pages.

In [20]:
import duckdb

con3 = duckdb.connect()
con3.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# Total distinct clients in the fact table, no filtering at all
total_clients_query = f"""
SELECT COUNT(DISTINCT client_hash_id) AS total_clients
FROM {FACT}
WHERE month IN ('2026-02', '2026-03')
"""
total_clients = con3.sql(total_clients_query).df()['total_clients'].iloc[0]

# Clients that survive Week 3's actual filter thresholds
filtered_clients_query = f"""
WITH prior AS (
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS gsc_impressions_prior30,
           SUM(gsc_clicks) AS gsc_clicks_prior30
    FROM {FACT}
    WHERE month = '2026-02'
    GROUP BY content_hash_id, client_hash_id
)
SELECT COUNT(DISTINCT client_hash_id) AS filtered_clients
FROM prior
WHERE gsc_impressions_prior30 >= 50 AND gsc_clicks_prior30 >= 3
"""
filtered_clients = con3.sql(filtered_clients_query).df()['filtered_clients'].iloc[0]

print(f"Total distinct clients in warehouse (Feb-Mar 2026, no filter): {total_clients}")
print(f"Clients with at least one page surviving the impressions/clicks filter: {filtered_clients}")
print(f"Clients in this notebook's actual df (Section 1, already loaded): {df['client_hash_id'].nunique()}")

# Rows per client, to see if it's genuinely 30 balanced clients or a few huge clients
# dominating a long tail of small ones — matters for whether more clients would actually help
rows_per_client = df.groupby('client_hash_id').size().sort_values(ascending=False)
print(f"\nRows per client (df, current notebook):")
print(rows_per_client.describe())
print(f"\nLargest client: {rows_per_client.iloc[0]} rows")
print(f"Smallest client: {rows_per_client.iloc[-1]} rows")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total distinct clients in warehouse (Feb-Mar 2026, no filter): 59
Clients with at least one page surviving the impressions/clicks filter: 32
Clients in this notebook's actual df (Section 1, already loaded): 30

Rows per client (df, current notebook):
count      30.000000
mean      983.866667
std      1931.944147
min         3.000000
25%        23.000000
50%       125.000000
75%       752.000000
max      7393.000000
dtype: float64

Largest client: 7393 rows
Smallest client: 3 rows


## 10. New feature category — GA4 engagement signals

The fact table contains GA4 engagement columns never used in any prior section:
ga4_sessions, ga4_engaged_sessions, ga4_total_engagement_sec. The FlyRank research paper's
Finding #5 found engagement and visibility consistency move together (+11.2 health points
between strongest and weakest engagement buckets), independent evidence this category may
carry signal.

Checked GA4 availability first, since client_has_ga4 exists specifically because coverage isn't
universal: within this notebook's 30 clients, roughly half have GA4 access (17 of ~35 client-
month combinations; note client_has_ga4 isn't perfectly constant per client across months in
this data, worth a one-line caveat but not a blocker). That's too large a gap to fillna(0) —
doing so would make "no GA4 access" indistinguishable from "zero engagement," silently biasing
the feature toward whatever clients happen to lack access.

Handling this properly: build engagement_rate_prior30 (= ga4_engaged_sessions / ga4_sessions,
prior30 window only, mirroring how gsc_ctr_prior30 was built) and leave it null where GA4 access
is absent, rather than zero-filling. Add a separate has_ga4_prior30 boolean feature so the model
can use "GA4 access exists or not" as its own signal if that's informative, distinct from the
engagement rate itself. This follows the same discipline as the missingness handling already in
place for word_count and content_type in Section 2 (dummy_na=True), extended to a case where the
missingness itself might carry real information rather than being pure noise to ignore.

In [24]:
import duckdb

con5 = duckdb.connect()
con5.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# How many of the 30 clients in df actually have GA4 access?
ga4_availability_query = f"""
SELECT client_has_ga4, COUNT(DISTINCT client_hash_id) AS client_count
FROM {FACT}
WHERE month = '2026-02'
GROUP BY client_has_ga4
"""
ga4_avail = con5.sql(ga4_availability_query).df()
print("GA4 access by client, Feb 2026 window (warehouse-wide, before filtering to df's 30 clients):")
print(ga4_avail)

# Now check specifically within the 30 clients already in df
client_list = "', '".join(df['client_hash_id'].unique())
ga4_in_df_query = f"""
SELECT client_has_ga4, COUNT(DISTINCT client_hash_id) AS client_count
FROM {FACT}
WHERE month = '2026-02' AND client_hash_id IN ('{client_list}')
GROUP BY client_has_ga4
"""
ga4_in_df = con5.sql(ga4_in_df_query).df()
print("\nGA4 access, restricted to the 30 clients already in this notebook's df:")
print(ga4_in_df)

query_ga4_features = f"""
SELECT
    content_hash_id, client_hash_id,
    SUM(ga4_sessions) AS ga4_sessions_prior30,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions_prior30,
    SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec_prior30,
    BOOL_OR(client_has_ga4) AS has_ga4_prior30
FROM {FACT}
WHERE month = '2026-02'
GROUP BY content_hash_id, client_hash_id
"""
ga4_df = con5.sql(query_ga4_features).df()

df_ga4 = df.merge(ga4_df, on=['content_hash_id', 'client_hash_id'], how='left')
print(f"Rows after merge: {len(df_ga4):,} (should still be {len(df):,})")

# Build engagement_rate_prior30 ONLY where GA4 sessions exist and are nonzero —
# leave it null everywhere else, do NOT fillna(0) here
df_ga4['engagement_rate_prior30'] = df_ga4.apply(
    lambda row: row['ga4_engaged_sessions_prior30'] / row['ga4_sessions_prior30']
    if row['has_ga4_prior30'] and row['ga4_sessions_prior30'] > 0 else pd.NA,
    axis=1
)

print(f"\nengagement_rate_prior30 non-null count: {df_ga4['engagement_rate_prior30'].notna().sum()} "
      f"out of {len(df_ga4)}")
print(f"has_ga4_prior30 value counts:\n{df_ga4['has_ga4_prior30'].value_counts(dropna=False)}")

# Sanity check: engagement_rate_prior30 should be null wherever has_ga4_prior30 is False,
# and only null there
mismatch = df_ga4[(df_ga4['has_ga4_prior30'] == True) & (df_ga4['engagement_rate_prior30'].isna())]
print(f"\nRows with GA4 access but still null engagement rate (likely zero sessions): {len(mismatch)}")

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

# Leakage sanity check on column naming — GA4 columns pulled only from month='2026-02',
# same prior-window discipline as every other feature, so no last30-window overlap possible
ga4_cols_check = ['ga4_sessions_prior30', 'ga4_engaged_sessions_prior30',
                   'ga4_total_engagement_sec_prior30', 'has_ga4_prior30', 'engagement_rate_prior30']
overlap_check_ga4 = [c for c in ga4_cols_check if 'last30' in c.lower()]
print(f"Columns with suspicious last30-window naming: {overlap_check_ga4} (should be empty)")

# Build extended feature set: original 5 numeric + has_ga4_prior30 (no missingness) +
# engagement_rate_prior30 (69% missing — filled with a sentinel, NOT zero, plus its own
# missingness flag so the model can tell "no signal" from "zero engagement")
df_ga4['engagement_rate_missing'] = df_ga4['engagement_rate_prior30'].isna().astype(int)
df_ga4['engagement_rate_prior30_filled'] = df_ga4['engagement_rate_prior30'].fillna(
    df_ga4['engagement_rate_prior30'].median()  # median of the KNOWN values only, not 0
)
df_ga4['has_ga4_prior30'] = df_ga4['has_ga4_prior30'].astype(int)

feature_cols_numeric_ga4 = feature_cols_numeric + [
    'has_ga4_prior30', 'engagement_rate_prior30_filled', 'engagement_rate_missing'
]
X_ga4 = pd.get_dummies(df_ga4[feature_cols_numeric_ga4 + feature_cols_categorical], dummy_na=True)
X_ga4[feature_cols_numeric_ga4] = X_ga4[feature_cols_numeric_ga4].fillna(0)

groups_ga4 = df_ga4['client_hash_id']
gss_ga4 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx_ga4, test_idx_ga4 = next(gss_ga4.split(X_ga4, y, groups=groups_ga4))
X_train_ga4, X_test_ga4 = X_ga4.iloc[train_idx_ga4], X_ga4.iloc[test_idx_ga4]
y_train_ga4, y_test_ga4 = y.iloc[train_idx_ga4], y.iloc[test_idx_ga4]

rf_ga4 = RandomForestClassifier(n_estimators=200, random_state=42)
rf_ga4.fit(X_train_ga4, y_train_ga4)

importance_ga4_df = pd.DataFrame({
    'feature': X_ga4.columns,
    'rf_importance': rf_ga4.feature_importances_
}).sort_values('rf_importance', ascending=False)

print("\nFeature importances with GA4 engagement features included:")
print(importance_ga4_df.head(15).to_string(index=False))
print("\nLeakage sanity check: none of the 3 new features should tower over the rest the way")
print("click_change_pct did (~0.52 importance alone).")

from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

scaler_ga4 = StandardScaler()
X_train_ga4_scaled = scaler_ga4.fit_transform(X_train_ga4)
X_test_ga4_scaled = scaler_ga4.transform(X_test_ga4)

logreg_ga4 = LogisticRegression(max_iter=1000, class_weight='balanced')
logreg_ga4.fit(X_train_ga4_scaled, y_train_ga4)
preds_ga4 = logreg_ga4.predict(X_test_ga4_scaled)
f1_ga4 = f1_score(y_test_ga4, preds_ga4)

print(f"Logistic Regression F1, grouped split, ORIGINAL 5 features: 0.350 (Section 2)")
print(f"Logistic Regression F1, grouped split, +GA4 engagement features: {round(f1_ga4, 3)}")
print(f"Change: {round(f1_ga4 - 0.350, 3)}")

perm_result_ga4 = permutation_importance(
    logreg_ga4, X_test_ga4_scaled, y_test_ga4, n_repeats=20, random_state=42, scoring='f1'
)
perm_ga4_df = pd.DataFrame({
    'feature': X_ga4.columns,
    'perm_importance_mean': perm_result_ga4.importances_mean,
    'perm_importance_std': perm_result_ga4.importances_std
}).sort_values('perm_importance_mean', ascending=False)

print("\nPermutation importance, extended GA4 feature set:")
print(perm_ga4_df.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

GA4 access by client, Feb 2026 window (warehouse-wide, before filtering to df's 30 clients):
   client_has_ga4  client_count
0           False            37
1            True            29

GA4 access, restricted to the 30 clients already in this notebook's df:
   client_has_ga4  client_count
0            True            17
1           False            18


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows after merge: 29,516 (should still be 29,516)

engagement_rate_prior30 non-null count: 9064 out of 29516
has_ga4_prior30 value counts:
has_ga4_prior30
False    20235
True      9281
Name: count, dtype: int64

Rows with GA4 access but still null engagement rate (likely zero sessions): 217
Columns with suspicious last30-window naming: [] (should be empty)


/tmp/ipykernel_413/4045888669.py:76: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_ga4['engagement_rate_prior30_filled'] = df_ga4['engagement_rate_prior30'].fillna(



Feature importances with GA4 engagement features included:
                       feature  rf_importance
      gsc_avg_position_prior30       0.225043
       gsc_impressions_prior30       0.217743
               gsc_ctr_prior30       0.212058
                    word_count       0.175782
            gsc_clicks_prior30       0.101122
engagement_rate_prior30_filled       0.022314
     main_intent_informational       0.009899
     main_intent_transactional       0.009240
        main_intent_commercial       0.008530
               has_ga4_prior30       0.006872
       engagement_rate_missing       0.006185
               main_intent_nan       0.002074
   content_type_feedly article       0.001304
  content_type_keyword article       0.001177
      main_intent_navigational       0.000573

Leakage sanity check: none of the 3 new features should tower over the rest the way
click_change_pct did (~0.52 importance alone).
Logistic Regression F1, grouped split, ORIGINAL 5 features: 0.350 (Secti

## 11. How stable is F1=0.350, really? — GroupKFold on the final model

Every result in this notebook has come from one specific 75/25 grouped split: 22 training
clients, 8 test clients, drawn once with GroupShuffleSplit(random_state=42). Section 9 showed
client sizes are extremely uneven (3 to 7,393 rows per client), and Section 8's tuning search
reported a GroupKFold cross-val score of ~0.28-0.29 for models very close to this one — noticeably
lower than the 0.350 this notebook has treated as the headline number throughout.

This section checks that directly: running the final 5-feature Logistic Regression (no GA4, no
age/freshness, no tuning — the exact model from Section 2) through 5-fold GroupKFold across the
entire df (not just the training portion, since this isn't a tuning search this time, it's a
direct estimate of how much the F1 number itself moves depending on which clients land in the
held-out fold).

This doesn't change the model or try to improve it — it answers a different question: is 0.350
a stable estimate of what this model does, or was it a somewhat fortunate result of exactly how
those particular 22 and 8 clients happened to divide.

In [25]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import numpy as np

gkf_final = GroupKFold(n_splits=5)

fold_results = []
fold_num = 1

for train_idx_k, test_idx_k in gkf_final.split(X, y, groups=groups):
    X_train_k, X_test_k = X.iloc[train_idx_k], X.iloc[test_idx_k]
    y_train_k, y_test_k = y.iloc[train_idx_k], y.iloc[test_idx_k]

    scaler_k = StandardScaler()
    X_train_k_scaled = scaler_k.fit_transform(X_train_k)
    X_test_k_scaled = scaler_k.transform(X_test_k)

    logreg_k = LogisticRegression(max_iter=1000, class_weight='balanced')
    logreg_k.fit(X_train_k_scaled, y_train_k)
    preds_k = logreg_k.predict(X_test_k_scaled)
    f1_k = f1_score(y_test_k, preds_k)

    train_clients_k = groups.iloc[train_idx_k].nunique()
    test_clients_k = groups.iloc[test_idx_k].nunique()

    fold_results.append({
        'fold': fold_num,
        'f1': round(f1_k, 3),
        'train_clients': train_clients_k,
        'test_clients': test_clients_k,
        'test_rows': len(test_idx_k)
    })
    fold_num += 1

fold_df = pd.DataFrame(fold_results)
print("5-fold GroupKFold, final 5-feature Logistic Regression:")
print(fold_df.to_string(index=False))

mean_f1 = fold_df['f1'].mean()
std_f1 = fold_df['f1'].std()

print(f"\nMean F1 across folds: {round(mean_f1, 3)}")
print(f"Std dev across folds: {round(std_f1, 3)}")
print(f"Range: {fold_df['f1'].min()} to {fold_df['f1'].max()}")
print(f"\nOriginal single-split F1 (Section 2, this notebook's headline number): 0.350")
print(f"Difference from cross-validated mean: {round(0.350 - mean_f1, 3)}")

5-fold GroupKFold, final 5-feature Logistic Regression:
 fold    f1  train_clients  test_clients  test_rows
    1 0.378             29             1       7393
    2 0.338             29             1       6033
    3 0.296             25             5       5362
    4 0.321             19            11       5364
    5 0.368             18            12       5364

Mean F1 across folds: 0.34
Std dev across folds: 0.034
Range: 0.296 to 0.378

Original single-split F1 (Section 2, this notebook's headline number): 0.350
Difference from cross-validated mean: 0.01


**Result: F1=0.350 is a stable, representative estimate, not a lucky draw — with one caveat
about fold composition.**

Mean F1 across 5 GroupKFold folds: 0.340, std dev 0.034, range 0.296 to 0.378. The original
single-split F1 of 0.350 sits just above the mean, well within the observed spread — a
reassuring result. This is a more defensible finding than Section 8's tuning-search cross-val
score of ~0.28-0.29, since that number came from only the training portion (22 clients) split
into folds, versus this section's folds drawn from the full 30-client df; the discrepancy is
likely explained by that difference in what was being folded, not by the earlier number being
wrong.

One real limitation in this specific cross-validation: GroupKFold balances folds by row count,
not by client count, so folds 1 and 2 each contain only a single test client (the largest ones,
7,393 and 6,033 rows), while folds 3-5 contain 5, 11, and 12 clients. Folds 1-2 are closer to
"how well does this model do on one specific large client" than a genuine multi-client average,
so the 5 fold scores aren't five equally-informative estimates of the same thing — the later
folds (3-5), with more clients each, are the more trustworthy read on general client-to-client
variation. Even accounting for that, the spread stays fairly tight (0.296 to 0.378), which
supports treating 0.350 as a fair, not favorable, representation of this model's real
performance.

**Final headline number for Week 7: F1=0.350 (single grouped split, Section 2), corroborated by
a 5-fold GroupKFold mean of 0.340 (std 0.034) — the two estimates agree closely enough that 0.350
can be reported as the honest number without a large caveat, beyond noting the natural fold-to-
fold variation documented here.**

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.